<a id="section1"></a>
## 1. Load Required Libraries

In [ ]:
# Import Essential Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import joblib

# SHAP for explainability
import shap

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ All libraries imported successfully!")
print(f"SHAP version: {shap.__version__}")

<a id="section2"></a>
## 2. Load Trained Models

In [ ]:
# Load models and preprocessing objects
models_dir = Path('models')

if not models_dir.exists():
    print("⚠ ERROR: Models directory not found!")
    print("Please run Notebook 1 first to train and save the models.")
else:
    print("="*80)
    print("LOADING TRAINED MODELS")
    print("="*80)
    
    # Load models
    xgb_model = joblib.load(models_dir / 'xgboost_model.pkl')
    elastic_model = joblib.load(models_dir / 'elastic_net_model.pkl')
    scaler = joblib.load(models_dir / 'scaler.pkl')
    feature_info = joblib.load(models_dir / 'feature_info.pkl')
    model_params = joblib.load(models_dir / 'model_parameters.pkl')
    
    print("✓ XGBoost model loaded")
    print("✓ Elastic Net model loaded")
    print("✓ Scaler loaded")
    print("✓ Feature information loaded")
    
    print(f"\nModel Information:")
    print(f"  - Number of features: {feature_info['n_features']}")
    print(f"  - Target transformation: {feature_info['target_transformation']}")
    print(f"  - XGBoost CV RMSE: {model_params['xgb_cv_rmse']}")
    print(f"  - Elastic Net CV RMSE: {model_params['elastic_cv_rmse']}")

<a id="section3"></a>
## 3. User Input for House Features

**Option 1:** Use a house from the test dataset  
**Option 2:** Manually input custom house features

In [ ]:
# Load test data to select a sample house
data_path = Path('./dataset')
test_df = pd.read_csv(data_path / 'test.csv')

print("="*80)
print("SELECT A HOUSE FOR PREDICTION")
print("="*80)
print(f"\nAvailable houses in test set: {len(test_df)}")
print("\nShowing first 5 houses:")
print(test_df[['Id', 'MSSubClass', 'LotArea', 'OverallQual', 'YearBuilt', 'GrLivArea']].head())

# Select a house by index (you can change this)
house_index = 0  # Change this to select different houses (0 to len(test_df)-1)

print(f"\n✓ Selected house at index {house_index} (ID: {test_df.iloc[house_index]['Id']})")

In [ ]:
# USER INPUT: Change house_index above or modify features manually here

# Display key features of the selected house
selected_house_id = test_df.iloc[house_index]['Id']

print("="*80)
print(f"SELECTED HOUSE DETAILS (ID: {selected_house_id})")
print("="*80)

# Show key features
key_features = ['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 
                'YearBuilt', 'YearRemodAdd', 'GrLivArea', 'TotalBsmtSF',
                'FullBath', 'HalfBath', 'BedroomAbvGr', 'TotRmsAbvGrd',
                'GarageCars', 'GarageArea', 'Neighborhood']

for feat in key_features:
    if feat in test_df.columns:
        value = test_df.iloc[house_index][feat]
        print(f"  {feat}: {value}")

print("\n✓ House selected. Ready for prediction.")

<a id="section4"></a>
## 4. Make Predictions

Predict house price using both Elastic Net and XGBoost models.

In [ ]:
# Function to preprocess and predict
def predict_single_house(house_data, xgb_model, elastic_model, scaler):
    """
    Predict house price for a single house.
    Note: This assumes house_data has already been through the same preprocessing
    pipeline as the training data (encoding, feature engineering, etc.)
    """
    # For demonstration, we'll load the preprocessed test data
    # In a real application, you'd need to apply all preprocessing steps
    
    results = {}
    
    # XGBoost prediction (no scaling needed)
    xgb_pred_log = xgb_model.predict(house_data)
    xgb_pred_price = np.expm1(xgb_pred_log[0])
    results['XGBoost'] = {
        'predicted_price': xgb_pred_price,
        'log_prediction': xgb_pred_log[0]
    }
    
    # Elastic Net prediction (needs scaling)
    house_scaled = scaler.transform(house_data)
    elastic_pred_log = elastic_model.predict(house_scaled)
    elastic_pred_price = np.expm1(elastic_pred_log[0])
    results['Elastic_Net'] = {
        'predicted_price': elastic_pred_price,
        'log_prediction': elastic_pred_log[0]
    }
    
    # Ensemble (average)
    avg_price = (xgb_pred_price + elastic_pred_price) / 2
    results['Ensemble'] = {
        'predicted_price': avg_price,
        'log_prediction': np.log1p(avg_price)
    }
    
    return results

print("✓ Prediction function defined")
print("\nNote: To use this function, you need preprocessed data.")
print("For quick demo, we'll show predictions on a validation sample from Notebook 1.")

In [ ]:
# DEMO: Load preprocessed data and make predictions
# In a real scenario, you would preprocess the selected house through the full pipeline

print("="*80)
print("HOUSE PRICE PREDICTIONS")
print("="*80)

# Note: For this demo, we need to load or recreate the preprocessed test data
# This would typically come from running the preprocessing pipeline

print("\nTo make predictions on the selected house:")
print("1. The house data must go through the same preprocessing as training data")
print("2. This includes missing value treatment, encoding, and feature engineering")
print("3. Then use the prediction function above")

print("\n✓ See Notebook 1 for the full preprocessing pipeline")
print("✓ For a complete end-to-end demo, run all cells in Notebook 1 first")

<a id="section5"></a>
## 5. SHAP Explanations

Use SHAP (SHapley Additive exPlanations) to explain individual predictions.

In [ ]:
# Create SHAP explainer for XGBoost
print("="*80)
print("INITIALIZING SHAP EXPLAINER")
print("="*80)

# Create explainer
explainer = shap.TreeExplainer(xgb_model)

print("✓ SHAP TreeExplainer created for XGBoost model")
print("\nSHAP will show:")
print("  - Which features increase the predicted price (push up)")
print("  - Which features decrease the predicted price (push down)")
print("  - By how much each feature contributes to the final prediction")

In [ ]:
# DEMO: Calculate SHAP values for a sample house
# Note: In practice, you would calculate this for your preprocessed house data

print("\n" + "="*80)
print("SHAP VALUE CALCULATION")
print("="*80)

print("\nTo calculate SHAP values for your house:")
print("""\n# Example code:
# Assuming 'house_features' is a preprocessed DataFrame with one row
shap_values = explainer.shap_values(house_features)

# Create SHAP explanation object
shap_exp = shap.Explanation(
    values=shap_values[0],  # SHAP values for the house
    base_values=explainer.expected_value,  # Base prediction
    data=house_features.iloc[0].values,  # Feature values
    feature_names=house_features.columns.tolist()  # Feature names
)
""")

print("\n✓ SHAP calculation code provided above")

In [ ]:
# Visualization: SHAP Waterfall Plot
print("\n" + "="*80)
print("SHAP VISUALIZATIONS")
print("="*80)

print("\nSHAP Waterfall Plot:")
print("  - Shows how each feature contributes to pushing the prediction")
print("  - From base value (average prediction) to final prediction")
print("  - Red bars = increase price, Blue bars = decrease price")

print("\n# Example code to create waterfall plot:")
print("""\nfig, ax = plt.subplots(figsize=(12, 8))
shap.plots.waterfall(shap_exp, max_display=15, show=False)
plt.title('SHAP Waterfall Plot - Feature Contributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
""")

print("\n✓ Waterfall plot code provided above")

In [ ]:
# Visualization: SHAP Force Plot
print("\nSHAP Force Plot:")
print("  - Shows features pushing prediction higher (red) or lower (blue)")
print("  - Width of each block = magnitude of impact")
print("  - Interactive visualization")

print("\n# Example code to create force plot:")
print("""\nfig, ax = plt.subplots(figsize=(14, 3))
shap.plots.force(
    explainer.expected_value,
    shap_values[0],
    house_features.iloc[0],
    matplotlib=True,
    show=False
)
plt.title('SHAP Force Plot - Push/Pull Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
""")

print("\n✓ Force plot code provided above")

<a id="section6"></a>
## 6. Interactive Feature Analysis

Analyze how changing specific features would affect the prediction.

In [ ]:
# Feature Impact Analysis
print("="*80)
print("FEATURE IMPACT ANALYSIS")
print("="*80)

print("\nHow to analyze feature impact:")
print("\n1. Get SHAP values for your house")
print("2. Sort features by absolute SHAP value")
print("3. Top features have the biggest impact on the prediction")

print("\n# Example code:")
print("""\n# Create DataFrame of feature impacts
feature_impact = pd.DataFrame({
    'Feature': house_features.columns,
    'Value': house_features.iloc[0].values,
    'SHAP_Value': shap_values[0],
    'Impact': ['Increases Price' if x > 0 else 'Decreases Price' for x in shap_values[0]]
})

# Sort by absolute SHAP value
feature_impact['Abs_SHAP'] = feature_impact['SHAP_Value'].abs()
feature_impact = feature_impact.sort_values('Abs_SHAP', ascending=False)

# Display top 15 features
print("\nTop 15 Most Important Features for This House:")
print(feature_impact[['Feature', 'Value', 'SHAP_Value', 'Impact']].head(15))
""")

print("\n✓ Feature impact analysis code provided")

In [ ]:
# What-If Analysis
print("\n" + "="*80)
print("WHAT-IF ANALYSIS")
print("="*80)

print("\nTo perform what-if analysis:")
print("\n1. Copy the house features")
print("2. Modify one or more features (e.g., increase OverallQual from 5 to 7)")
print("3. Run prediction on the modified house")
print("4. Compare the predictions")

print("\n# Example code:")
print("""\n# Original house
original_price = predictions['XGBoost']['predicted_price']

# Modify house (e.g., improve overall quality)
modified_house = house_features.copy()
if 'OverallQual' in modified_house.columns:
    modified_house['OverallQual'] = modified_house['OverallQual'] + 1

# Predict modified house
modified_pred = xgb_model.predict(modified_house)
modified_price = np.expm1(modified_pred[0])

# Compare
price_diff = modified_price - original_price
price_diff_pct = (price_diff / original_price) * 100

print(f"Original Price: ${original_price:,.0f}")
print(f"Modified Price: ${modified_price:,.0f}")
print(f"Difference: ${price_diff:,.0f} ({price_diff_pct:.1f}%)")
""")

print("\n✓ What-if analysis code provided")

## Summary and Recommendations

This notebook provides tools for:
1. **Single house price prediction** using trained models
2. **SHAP explanations** to understand why the model made its prediction
3. **Feature impact analysis** to identify which features matter most
4. **What-if analysis** to explore how changes affect the price

### To use this notebook effectively:

1. **First, run Notebook 1** to train models and save them
2. **Prepare your house data** with the same preprocessing as training
3. **Use the prediction function** to get price estimates
4. **Generate SHAP explanations** to understand the prediction
5. **Perform what-if analysis** to explore scenarios

### Key Insights from SHAP:

- **Positive SHAP values** → Feature increases the predicted price
- **Negative SHAP values** → Feature decreases the predicted price
- **Magnitude** → How much the feature impacts the prediction
- **Base value** → Average model prediction across all houses

---

**Next Steps:**
- Use Notebook 3 to generate predictions for the entire test set
- Experiment with different houses to understand model behavior
- Use what-if analysis to provide recommendations to homeowners